# Lag "fasit" for bokmål-nynorsk text alignment basert på lånekassens dokumenter
Lånekassen har masse paralelldata 

In [ ]:
from pathlib import Path
import pandas as pd
from hemmelig import data_path_2021

source_p = Path(data_path_2021)
out_p = Path("output/lånekassen_data.json")

if not out_p.exists():
    filer = [e for e in source_p.iterdir() if "lanekassen" in e.name]

    dfs = []
    for e in filer:
        with e.open("rb") as f:
            dfs.append(pd.read_json(e, lines=True))
    df = pd.concat(dfs)
    df.index = range(len(df))
    df.to_json(out_p, index=False)
df = pd.read_json(out_p)
df

In [ ]:
nynorske = df[df.lang == "nno"]
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"]
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

# Filtrer på språkkode i url

In [ ]:
nynorske_url = nynorske[nynorske.url.apply(lambda x: "nn-NO" in x)]
bokmålske_url = bokmålske[bokmålske.url.apply(lambda x: "nb-NO" in x)]


## Tekster med omvendt språkkode i url

In [ ]:
nynorske[nynorske.url.apply(lambda x: "nb-NO" in x)]


In [ ]:
bokmålske[bokmålske.url.apply(lambda x: "nn-NO" in x)]

# Finn par av urler der kun språkkoden er ulik

In [ ]:
from collections import defaultdict

def lik_utenom_språkkode(nn_url, bm_url):
    biter = nn_url.split("nn-NO")
    biter2 = bm_url.split("nb-NO")
    return biter == biter2

par = defaultdict(list)

for e in nynorske_url.itertuples():
    for e2 in bokmålske_url.itertuples():
        if lik_utenom_språkkode(e.url, e2.url):
            par["nynorsk_doc_hash"].append(e.doc_hash)
            par["bokmål_doc_hash"].append(e2.doc_hash)
            par["nynorsk_url"].append(e.url)
            par["bokmål_url"].append(e2.url)


par_df = pd.DataFrame(par)
par_df.to_csv("lanekassen_fasit.csv", index=False)


In [ ]:
import pprint

for e in par_df.itertuples():
    pprint.pprint(list(nynorske[nynorske.doc_hash==e.nynorsk_doc_hash].fulltext)[0][:3])
    pprint.pprint(list(bokmålske[bokmålske.doc_hash==e.bokmål_doc_hash].fulltext)[0][:3])
    print("\n________________________________\n")